# 01 — Setup & Verification

This notebook verifies that all components work:
1. MGSM dataset loads correctly for all 5 languages
2. Gemma 3 4B IT model loads and generates answers
3. Gemma Scope 2 SAEs load via SAELens
4. nnsight can extract activations
5. Baseline accuracy on a small subset

**Run this on Colab with A100 GPU.**

## 0. Install Dependencies

In [ ]:
# Run this cell on Colab, then RESTART RUNTIME before continuing
# (Runtime → Restart session)
!pip install -q torch transformers "datasets==2.21.0" sae-lens nnsight python-dotenv tqdm scikit-learn sentence-transformers

In [ ]:
import sys
import os

# Set HF token from Colab secrets
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print(f"HF_TOKEN set: {'yes' if os.environ.get('HF_TOKEN') else 'NO — add it to Colab secrets!'}")

# Add src/ to path if needed
if 'src' not in sys.path:
    sys.path.insert(0, '.')

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load MGSM Dataset

In [ ]:
from datasets import load_dataset

TARGET_LANGUAGES = ['en', 'zh', 'es', 'bn', 'sw']

mgsm_data = {}
for lang in TARGET_LANGUAGES:
    ds = load_dataset('juletxara/mgsm', lang, split='test', trust_remote_code=True)
    mgsm_data[lang] = ds
    print(f"{lang}: {len(ds)} examples")
    # Show first example
    print(f"  Q: {ds[0]['question'][:80]}...")
    print(f"  A: {ds[0]['answer_number']}")
    print()

In [ ]:
# Verify the same problem appears in all languages (problem 0)
print("Problem 0 across languages:")
for lang in TARGET_LANGUAGES:
    print(f"\n[{lang}] answer={mgsm_data[lang][0]['answer_number']}")
    print(f"  {mgsm_data[lang][0]['question'][:120]}")

## 2. Load Gemma 3 4B IT

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'google/gemma-3-4b-it'
token = os.environ.get('HF_TOKEN')
assert token, "Set HF_TOKEN environment variable first!"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)

print("Loading model (BF16)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=token,
)
model.eval()

print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"Model device: {model.device}")
print(f"Number of layers: {len(model.model.layers)}")
print(f"Hidden size: {model.config.hidden_size}")

In [ ]:
# Quick test: generate on a simple English MGSM problem
test_q = mgsm_data['en'][0]['question']
test_gold = mgsm_data['en'][0]['answer_number']

messages = [
    {'role': 'user', 'content': test_q},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(f"Prompt ({len(prompt)} chars):")
print(prompt[:300])
print("...")

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    gen_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)

output = tokenizer.decode(gen_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"\nModel output:\n{output}")
print(f"\nGold answer: {test_gold}")

In [ ]:
# Test answer parsing
import re

def parse_answer_number(text):
    """Extract final numeric answer from model output."""
    text = text.strip()
    # Pattern: #### <number>
    match = re.search(r'####\s*([\d,]+(?:\.\d+)?)', text)
    if match:
        return float(match.group(1).replace(',', ''))
    # Pattern: "The answer is <number>"
    for pattern in [
        r'[Tt]he answer is\s*([\d,]+(?:\.\d+)?)',
        r'[Aa]nswer[:\s]*([\d,]+(?:\.\d+)?)',
    ]:
        match = re.search(pattern, text)
        if match:
            return float(match.group(1).replace(',', ''))
    # Fallback: last number
    numbers = re.findall(r'[\d,]+(?:\.\d+)?', text)
    if numbers:
        return float(numbers[-1].replace(',', ''))
    return None

parsed = parse_answer_number(output)
print(f"Parsed: {parsed}, Gold: {test_gold}, Correct: {parsed == test_gold}")

## 3. Load Gemma Scope 2 SAE

In [ ]:
from sae_lens import SAE

# First, let's check what SAE IDs are available
# The exact naming convention needs verification
SAE_RELEASE = 'google/gemma-scope-2-4b-it-res'

# Try loading a 16k SAE at layer 9
# The sae_id format may be: layer_9/width_16k/average_l0_XX
# We need to check the actual repo structure

print(f"Attempting to load SAE from {SAE_RELEASE}...")
print("Note: The exact sae_id format needs to be verified against the HF repo.")
print("If loading fails, check: https://huggingface.co/google/gemma-scope-2-4b-it-res")

In [ ]:
# List available SAEs in the release
# SAELens may have a method for this, or we check the YAML
try:
    from huggingface_hub import list_repo_tree
    files = list(list_repo_tree(SAE_RELEASE, repo_type='model'))
    # Show directory structure
    dirs = set()
    for f in files:
        parts = f.rfilename.split('/')
        if len(parts) >= 2:
            dirs.add('/'.join(parts[:2]))
    print("Available SAE directories (first 20):")
    for d in sorted(dirs)[:20]:
        print(f"  {d}")
except Exception as e:
    print(f"Could not list repo: {e}")
    print("Try browsing https://huggingface.co/google/gemma-scope-2-4b-it-res manually")

In [ ]:
# Try loading an SAE (adjust sae_id based on repo structure)
try:
    sae, cfg_dict, sparsity = SAE.from_pretrained(
        release=SAE_RELEASE,
        sae_id='layer_9/width_16k/average_l0_50',  # May need adjustment
        device='cuda',
    )
    print(f"SAE loaded successfully!")
    print(f"  d_in (model dim): {sae.cfg.d_in}")
    print(f"  d_sae (features): {sae.cfg.d_sae}")
    print(f"  W_enc shape: {sae.W_enc.shape}")
    print(f"  W_dec shape: {sae.W_dec.shape}")
except Exception as e:
    print(f"Failed to load SAE: {e}")
    print("\nTrying alternative sae_id formats...")
    # Try different naming patterns
    for sae_id in [
        'layer_9/width_16k/average_l0_10',
        'layer_9/width_16k/average_l0_100',
        'layer_9/width_16k/l0_medium',
        'layer_9/width_16384/average_l0_50',
    ]:
        try:
            sae, cfg_dict, sparsity = SAE.from_pretrained(
                release=SAE_RELEASE,
                sae_id=sae_id,
                device='cuda',
            )
            print(f"  SUCCESS with sae_id='{sae_id}'")
            print(f"  d_sae={sae.cfg.d_sae}, W_dec={sae.W_dec.shape}")
            break
        except Exception as e2:
            print(f"  Failed: {sae_id} -> {e2}")

In [ ]:
# Test SAE encoding/decoding
if 'sae' in dir():
    # Create a dummy activation (d_model=2560)
    dummy = torch.randn(1, 2560, device='cuda', dtype=torch.float32)
    
    with torch.no_grad():
        features = sae.encode(dummy)
        reconstruction = sae.decode(features)
    
    print(f"Input shape: {dummy.shape}")
    print(f"Features shape: {features.shape}")
    print(f"Reconstruction shape: {reconstruction.shape}")
    print(f"Active features (nonzero): {(features > 0).sum().item()} / {features.shape[-1]}")
    print(f"Reconstruction MSE: {((dummy - reconstruction) ** 2).mean().item():.6f}")
else:
    print("SAE not loaded — fix the loading step above first.")

## 4. Test nnsight Activation Extraction

In [ ]:
from nnsight import NNsight

nn_model = NNsight(model)

# Extract residual stream at layer 9 for a single input
test_input = tokenizer("What is 2 + 3?", return_tensors='pt').to(model.device)

with nn_model.trace(test_input, scan=False, validate=False):
    resid_9 = nn_model.model.layers[9].output[0].save()
    resid_17 = nn_model.model.layers[17].output[0].save()
    resid_29 = nn_model.model.layers[29].output[0].save()

print(f"Layer 9 residual shape: {resid_9.value.shape}")   # (1, seq_len, 2560)
print(f"Layer 17 residual shape: {resid_17.value.shape}")
print(f"Layer 29 residual shape: {resid_29.value.shape}")
print(f"\nActivation norms:")
print(f"  Layer 9:  {resid_9.value.float().norm(dim=-1).mean().item():.2f}")
print(f"  Layer 17: {resid_17.value.float().norm(dim=-1).mean().item():.2f}")
print(f"  Layer 29: {resid_29.value.float().norm(dim=-1).mean().item():.2f}")

In [ ]:
# Test: encode real activations through SAE
if 'sae' in dir():
    last_token_act = resid_9.value[:, -1, :].float()  # (1, 2560)
    with torch.no_grad():
        features = sae.encode(last_token_act.to('cuda'))
    
    print(f"Feature activations shape: {features.shape}")
    n_active = (features > 0).sum().item()
    print(f"Active features: {n_active} / {features.shape[-1]} ({100*n_active/features.shape[-1]:.1f}%)")
    
    top_features = torch.argsort(features[0], descending=True)[:10]
    print(f"\nTop 10 active features:")
    for idx in top_features:
        print(f"  Feature {idx.item()}: activation = {features[0, idx].item():.4f}")
else:
    print("SAE not loaded.")

## 5. Quick Baseline: Run on 5 problems per language

In [ ]:
# Run model on first 5 problems per language to verify pipeline
N_TEST = 5

results = {}
for lang in TARGET_LANGUAGES:
    predictions = []
    gold_answers = []
    
    for i in range(N_TEST):
        q = mgsm_data[lang][i]['question']
        gold = mgsm_data[lang][i]['answer_number']
        gold_answers.append(gold)
        
        messages = [{'role': 'user', 'content': q}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        
        with torch.no_grad():
            gen_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)
        
        output = tokenizer.decode(
            gen_ids[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        parsed = parse_answer_number(output)
        predictions.append(parsed)
        
        correct = parsed is not None and abs(parsed - gold) < 1e-6
        print(f"[{lang}] Q{i}: gold={gold}, parsed={parsed}, correct={correct}")
    
    correct_count = sum(
        1 for p, g in zip(predictions, gold_answers)
        if p is not None and abs(p - g) < 1e-6
    )
    results[lang] = correct_count / N_TEST

print("\n=== Quick Baseline Results ===")
for lang, acc in results.items():
    print(f"  {lang}: {acc:.0%} ({int(acc * N_TEST)}/{N_TEST})")
print(f"  Average: {sum(results.values()) / len(results):.0%}")

## 6. Memory Usage Report

In [ ]:
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Memory Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"GPU Memory Total:     {total / 1e9:.2f} GB")
    print(f"GPU Memory Free:      {(total - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

## Summary

After running this notebook, confirm:
- [ ] MGSM loads for all 5 languages (250 problems each)
- [ ] Same problems appear in same order across languages
- [ ] Gemma 3 4B IT loads in BF16
- [ ] Model generates reasonable math answers
- [ ] Answer parser extracts correct numbers
- [ ] SAE loads from Gemma Scope 2 (note the exact sae_id that works)
- [ ] SAE encoding produces sparse features
- [ ] nnsight extracts residual stream activations
- [ ] Real activations encode through SAE correctly
- [ ] Memory usage is manageable on A100

**Record the exact working `sae_id` format** — we need this for all subsequent experiments.